# Guardrails: Input Sanitization, Output Validation, and Safety Filters

This notebook demonstrates simple application-level checks around a fictional text service. It uses harmless strings and does not connect to an LLM.

**Learning goals**
- Validate input shape and size.
- Treat sanitization as one layer, not a substitute for authorization.
- Validate structured output against a schema and enforce a simple safety rule.

In [ ]:
MAX_LENGTH = 200
ALLOWED_CATEGORIES = {"billing", "technical", "general"}

def validate_input(text):
    if not isinstance(text, str):
        return False, "Input must be text."
    if not text.strip():
        return False, "Input cannot be empty."
    if len(text) > MAX_LENGTH:
        return False, "Input is too long."
    return True, text.strip()

for sample in ("Help with my invoice", "   ", "x" * 250):
    print(validate_input(sample))

## 1. Normalize and constrain input

Normalization can remove control characters and enforce length limits. Avoid destructive transformations that alter meaning; for HTML, SQL, shell, and other contexts, use context-specific encoders and parameterized APIs instead of generic string filtering.

In [ ]:
def normalize_text(text):
    # Keep printable characters and normalize surrounding whitespace.
    return "".join(char for char in text if char.isprintable()).strip()

raw = "  Please review my invoice.\x00  "
ok, value = validate_input(raw)
print("Accepted:", ok, "Normalized:", normalize_text(value) if ok else value)

## 2. Validate output structure and content

Suppose a classifier returns a dictionary. Validate required fields, types, allowed values, and unsafe content before using or displaying it.

In [ ]:
def validate_result(result):
    if not isinstance(result, dict):
        return False, "Result must be an object."
    if set(result) != {"category", "reply"}:
        return False, "Unexpected or missing fields."
    if result["category"] not in ALLOWED_CATEGORIES:
        return False, "Unknown category."
    if not isinstance(result["reply"], str) or len(result["reply"]) > 300:
        return False, "Reply must be short text."
    if "password" in result["reply"].lower():
        return False, "Reply contains disallowed sensitive-data wording."
    return True, "Accepted."

examples = [
    {"category": "billing", "reply": "I can help review the invoice."},
    {"category": "admin", "reply": "Granted."},
    {"category": "general", "reply": "Please send your password."},
]
for item in examples:
    print(validate_result(item))

## 3. Layer checks with permissions and escalation

Filters can miss context and can block benign content. Sensitive actions should be governed by explicit permissions and a human escalation path, not only by keyword rules.

In [ ]:
def may_perform_action(action, authenticated, confirmed):
    allowed = {"show_public_help", "update_account_email"}
    if action not in allowed:
        return False, "Action is not allowlisted."
    if not authenticated:
        return False, "Authentication required."
    if action == "update_account_email" and not confirmed:
        return False, "User confirmation required."
    return True, "Action may proceed."

print(may_perform_action("show_public_help", True, False))
print(may_perform_action("update_account_email", True, False))

## Takeaways

- Validate at system boundaries: inputs, model outputs, tool requests, and final actions.
- Prefer allowlists and schema checks; use escaping or parameterization appropriate to the output context.
- Keep authorization separate from model-generated text.
- Test edge cases and monitor false positives and false negatives; no simple filter is complete.